## 1. Important Imports

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd 
import numpy as np 
import seaborn as se 
import matplotlib as plt
import random 

## 2. Createting PySpark Session

In [0]:
spark = SparkSession.builder.appName('Data_Analysis_with_PySpark').getOrCreate()

## 3. Generating Data

In [0]:
names = [
    "Alice", "Bob", "Charlie", "David", "Eve", "Fiona", "George", "Hannah",
    "Ivy", "Jack", "Kaitlyn", "Liam", "Olivia", "Liam", "Emma", "Noah", 
    "Ava", "Oliver", "Charlotte", "Elijah", "Sophia", "James", "Amelia", 
    "Benjamin", "Isabella", "Lucas", "Mia", "Mason", "Harper", "Ethan", 
    "Evelyn", "Alexander", "Abigail", "Henry", "Ella", "Jackson", "Scarlett", 
    "Aiden", "Grace", "Samuel", "Lily", "Sebastian"
]
genders = ["Male", "Female", None]
subjects = ["Math", "Science", "History", "English", "Art", "PE", None]
cities = [
    "New York", "Los Angeles", "Chicago", "Houston", 
    "Bangalore", "Hajipur", "Sitamardhi", "MP", None
]
states = ["NY", "CA", "IL", "TX", "Bihar", "Karnataka", "Sitamardhi", None]
countries = ["USA", "India", "Pakistan", "Nepal", "China", None]
graduated_status = ["Yes", "No", None]

data = [
    (
        i, 
        random.choice(names),  # student_name
        random.choice([random.randint(18, 25), None]),  # age
        random.choice(genders),  # gender
        random.choice(subjects),  # subject
        random.choice([random.randint(50, 100), None]),  # marks
        random.choice(cities),  # city
        random.choice(states),  # state
        random.choice(countries),  # country
        random.choice(graduated_status),  # graduated
    )
    for i in range(1, 501)
]

## 4. Creating a DATAFRAME

In [0]:
schema = StructType([
    StructField("student_id", IntegerType(), True),
    StructField("student_name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("subject", StringType(), True),
    StructField("marks", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True),
    StructField("graduated", StringType(), True),
])

df = spark.createDataFrame(data,schema=schema)
df.show(5)

+----------+------------+----+------+-------+-----+----------+---------+--------+---------+
|student_id|student_name| age|gender|subject|marks|      city|    state| country|graduated|
+----------+------------+----+------+-------+-----+----------+---------+--------+---------+
|         1|   Sebastian|null|Female|History|   77| Bangalore|    Bihar|   India|      Yes|
|         2|      Amelia|  25|Female|English|   90|      null|       NY|   Nepal|     null|
|         3|      Amelia|  24|  Male|     PE| null|   Hajipur|       NY|     USA|     null|
|         4|        Lily|  21|  null|     PE|   58|      null|       CA|   India|     null|
|         5|     Abigail|  19|  null|English| null|Sitamardhi|Karnataka|Pakistan|       No|
+----------+------------+----+------+-------+-----+----------+---------+--------+---------+
only showing top 5 rows



## 5.OverView of DataFrame

In [0]:
df.show(10)

+----------+------------+----+------+-------+-----+----------+----------+--------+---------+
|student_id|student_name| age|gender|subject|marks|      city|     state| country|graduated|
+----------+------------+----+------+-------+-----+----------+----------+--------+---------+
|         1|   Sebastian|null|Female|History|   77| Bangalore|     Bihar|   India|      Yes|
|         2|      Amelia|  25|Female|English|   90|      null|        NY|   Nepal|     null|
|         3|      Amelia|  24|  Male|     PE| null|   Hajipur|        NY|     USA|     null|
|         4|        Lily|  21|  null|     PE|   58|      null|        CA|   India|     null|
|         5|     Abigail|  19|  null|English| null|Sitamardhi| Karnataka|Pakistan|       No|
|         6|   Alexander|null|Female|    Art| null| Bangalore| Karnataka|    null|       No|
|         7|      Sophia|null|Female|History|   50|   Houston|Sitamardhi|     USA|      Yes|
|         8|        Ella|  22|  Male|English|   55|        MP|        

In [0]:
df.describe().show()

+-------+-----------------+------------+------------------+------+-------+------------------+----------+-----+-------+---------+
|summary|       student_id|student_name|               age|gender|subject|             marks|      city|state|country|graduated|
+-------+-----------------+------------+------------------+------+-------+------------------+----------+-----+-------+---------+
|  count|              500|         500|               256|   332|    421|               252|       444|  442|    424|      335|
|   mean|            250.5|        null|       21.50390625|  null|   null| 73.73412698412699|      null| null|   null|     null|
| stddev|144.4818327679989|        null|2.2016002145943667|  null|   null|14.343701715947722|      null| null|   null|     null|
|    min|                1|     Abigail|                18|Female|    Art|                50| Bangalore|Bihar|  China|       No|
|    max|              500|      Sophia|                25|  Male|Science|               100|Sita

In [0]:
df.dtypes

Out[7]: [('student_id', 'int'),
 ('student_name', 'string'),
 ('age', 'int'),
 ('gender', 'string'),
 ('subject', 'string'),
 ('marks', 'int'),
 ('city', 'string'),
 ('state', 'string'),
 ('country', 'string'),
 ('graduated', 'string')]

In [0]:
df.printSchema()

root
 |-- student_id: integer (nullable = true)
 |-- student_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- subject: string (nullable = true)
 |-- marks: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- graduated: string (nullable = true)



In [0]:
## Checking Missing Values
str_column = ['student_name','gender','subject','city','state','country','graduated']
num_column = ['student_id','age','marks']

missing_val_dict = {}
for index,column in enumerate(df.columns):
    if column in str_column:
        missing_str_count = df.filter(col(column).eqNullSafe(None)\
            | col(column).isNull()
            ).count()
        missing_val_dict.update({column:missing_str_count})
    if column in num_column:
        missing_num_count = df.where(col(column).isin([0,None,np.nan])).count()
        missing_val_dict.update({column:missing_num_count})
missing_df = pd.DataFrame.from_dict([missing_val_dict])
missing_df




,student_id,student_name,age,gender,subject,marks,city,state,country,graduated
0,0,0,0,168,79,0,56,58,76,165


## 6. Replacing Null Values

In [0]:
df = df.fillna({
    'gender' : 'UniSex',
    'subject' : 'Hindi',
    'city' : 'Kalitand',
    'State' : 'Others',
    'Country' : 'India',
    'graduated' : 'Failed'
})

In [0]:
## Checking Missing Values
str_column = ['student_name','gender','subject','city','state','country','graduated']
num_column = ['student_id','age','marks']

missing_val_dict = {}
for index,column in enumerate(df.columns):
    if column in str_column:
        missing_str_count = df.filter(col(column).eqNullSafe(None)\
            | col(column).isNull()
            ).count()
        missing_val_dict.update({column:missing_str_count})
    if column in num_column:
        missing_num_count = df.where(col(column).isin([0,None,np.nan])).count()
        missing_val_dict.update({column:missing_num_count})
missing_df = pd.DataFrame.from_dict([missing_val_dict])
missing_df




,student_id,student_name,age,gender,subject,marks,city,state,country,graduated
0,0,0,0,0,0,0,0,0,0,0


## 7. Checking duplicate Values in each column

In [0]:
for index,column in enumerate(df.columns):
    print(f"checking duplicates in the column: {column}")
    duplicate_count = df.groupBy(column).count().filter("count > 1 ")
    duplicate_count.show()


checking duplicates in the column: student_id
+----------+-----+
|student_id|count|
+----------+-----+
+----------+-----+

checking duplicates in the column: student_name
+------------+-----+
|student_name|count|
+------------+-----+
|       Lucas|    9|
|       Grace|   13|
|       James|   14|
|      Hannah|   12|
|        Jack|   11|
|         Ava|   11|
|        Ella|   11|
|      Evelyn|   11|
|        Noah|   12|
|       Mason|    9|
|       Ethan|   14|
|     Charlie|   16|
|         Bob|   11|
|        Liam|   15|
|   Sebastian|   12|
|     Jackson|   15|
|      Elijah|   16|
|   Alexander|   11|
|       Aiden|   14|
|       Alice|    9|
+------------+-----+
only showing top 20 rows

checking duplicates in the column: age
+----+-----+
| age|count|
+----+-----+
|  22|   39|
|null|  244|
|  20|   30|
|  19|   37|
|  23|   22|
|  25|   24|
|  24|   43|
|  21|   36|
|  18|   25|
+----+-----+

checking duplicates in the column: gender
+------+-----+
|gender|count|
+------+-----+
|Fe

In [0]:
duplicate_counts = []
# Loop through each column in the DataFrame
for column in df.columns:
    # Group by the column and count duplicates (count > 1)
    dup_count = df.groupBy(column).count().filter("count > 1")
    # Get the number of duplicate values
    count = dup_count.count()
    # Append the result as a tuple (column name, duplicate count)
    duplicate_counts.append((column, count))
# Convert the list to a Pandas DataFrame
dup_df = pd.DataFrame(duplicate_counts, columns=['Column_Name', "Dup_count"])
# Show the Pandas DataFrame
dup_df


,Column_Name,Dup_count
0,student_id,0
1,student_name,41
2,age,9
3,gender,3
4,subject,7
5,marks,52
6,city,9
7,state,8
8,country,5
9,graduated,3


In [0]:
# Loop through each column in the DataFrame
for column in df.columns:
    print(column)
    dup_count = df.groupBy(column).count().filter("count > 1")
    # Get the number of duplicate values
    count = dup_count.count()
    # Append the result as a tuple (column name, duplicate count)
    duplicate_counts.append((column, count))
# Convert the list to a Pandas DataFrame
dup_df = pd.DataFrame(duplicate_counts, columns=['Column_Name', "Dup_count"])
# Show the Pandas DataFrame
dup_df


student_id
student_name
age
gender
subject
marks
city
state
country
graduated


,Column_Name,Dup_count
0,student_id,0
1,student_name,41
2,age,9
3,gender,3
4,subject,7
5,marks,52
6,city,9
7,state,8
8,country,5
9,graduated,3


## 8.Descriptive Statistics and Basic Summarization

In [0]:
df.printSchema()

root
 |-- student_id: integer (nullable = true)
 |-- student_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = false)
 |-- subject: string (nullable = false)
 |-- marks: integer (nullable = true)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- country: string (nullable = false)
 |-- graduated: string (nullable = false)



In [0]:
# What are the central tendencies (mean, median, mode) of marks and age?
df.groupBy("subject").agg(
    mean("marks").alias("marks_mean"),
    mean("age").alias("mean_age")
).show()



+-------+-----------------+------------------+
|subject|       marks_mean|          mean_age|
+-------+-----------------+------------------+
|Science|74.68571428571428|21.846153846153847|
|    Art|74.52941176470588|            21.625|
|   Math|69.27777777777777| 21.15909090909091|
|English|75.97368421052632|21.952380952380953|
|History|           70.125|             21.75|
|  Hindi| 76.4390243902439| 20.81578947368421|
|     PE|74.27777777777777|21.454545454545453|
+-------+-----------------+------------------+



In [0]:
# What is the spread of marks in each subject?
df.groupBy("subject").agg(
    min("marks").alias("min_marks"),
    max("marks").alias("max_marks"),
).show()

+-------+---------+---------+
|subject|min_marks|max_marks|
+-------+---------+---------+
|Science|       52|       98|
|    Art|       51|      100|
|   Math|       50|      100|
|English|       53|      100|
|History|       50|      100|
|  Hindi|       51|      100|
|     PE|       51|      100|
+-------+---------+---------+



In [0]:
# What is the standard deviation of marks for different cities or countries?
df.groupBy("city").agg(
    stddev("marks").alias("stddev_marks")
).show()

+-----------+------------------+
|       city|      stddev_marks|
+-----------+------------------+
|  Bangalore|14.610065810546704|
|Los Angeles|16.024109328511205|
| Sitamardhi|12.218969915932014|
|    Chicago|13.553733939535025|
|    Hajipur|14.442752627123262|
|    Houston| 14.76028976159187|
|   New York|14.066032165097775|
|         MP|14.719827897544636|
|   Kalitand|14.922075819878197|
+-----------+------------------+



In [0]:
from pyspark.sql import functions as F

# Calculate the distribution of students in each subject
df.groupBy("subject").agg(
    F.count("student_id").alias("stu_count")  # Count the number of students in each subject
).show()


+-------+---------+
|subject|stu_count|
+-------+---------+
|Science|       76|
|    Art|       62|
|   Math|       76|
|English|       77|
|History|       58|
|  Hindi|       79|
|     PE|       72|
+-------+---------+



In [0]:
# How many students belong to each gender and how does their academic performance differ?
df.groupBy("gender").agg(
    F.count("*").alias("stu_count"),       # Count of students per gender
    F.avg("marks").alias("avg_marks"),     # Average marks per gender
    F.stddev("marks").alias("std_marks")   # Standard deviation of marks per gender
).show()


+------+---------+-----------------+------------------+
|gender|stu_count|        avg_marks|         std_marks|
+------+---------+-----------------+------------------+
|Female|      183|73.31428571428572|14.580884671452477|
|UniSex|      168|72.25333333333333|13.463758494298418|
|  Male|      149|75.88888888888889|14.822475849106302|
+------+---------+-----------------+------------------+



In [0]:
# What is the distribution of students across different cities, states, and countries?
df.groupBy("city","state","country").count().show()

+-----------+----------+--------+-----+
|       city|     state| country|count|
+-----------+----------+--------+-----+
|    Houston|Sitamardhi|     USA|    2|
|    Chicago|     Bihar|   China|    1|
|         MP|        NY|   India|    1|
|   New York| Karnataka|   India|    5|
|   Kalitand|Sitamardhi|Pakistan|    2|
|  Bangalore|Sitamardhi|     USA|    1|
|Los Angeles|        TX|   China|    2|
| Sitamardhi|        NY|   India|    4|
|    Hajipur|        TX|Pakistan|    4|
|Los Angeles|        CA|   China|    1|
|  Bangalore|     Bihar|     USA|    2|
|   New York|Sitamardhi|     USA|    2|
| Sitamardhi| Karnataka|   Nepal|    1|
|         MP|        IL|   India|    6|
| Sitamardhi| Karnataka|   India|    3|
|    Hajipur|        TX|     USA|    2|
|   New York|        IL|   Nepal|    2|
|    Chicago|Sitamardhi|   India|    2|
| Sitamardhi|     Bihar|   Nepal|    2|
| Sitamardhi|    Others|     USA|    2|
+-----------+----------+--------+-----+
only showing top 20 rows



In [0]:
# How many students are marked as graduated versus those who are not? What is the graduation rate?
graduated_status = df.groupBy("graduated").agg(F.count("*").alias("stu_count"))
total_stu = df.count()

graduated_status.withColumn("graducation_perc",col("stu_count")/total_stu * 100).show()


+---------+---------+----------------+
|graduated|stu_count|graducation_perc|
+---------+---------+----------------+
|   Failed|      165|            33.0|
|       No|      158|            31.6|
|      Yes|      177|            35.4|
+---------+---------+----------------+



In [0]:
graduated_status.show()

+---------+---------+
|graduated|stu_count|
+---------+---------+
|   Failed|      165|
|       No|      158|
|      Yes|      177|
+---------+---------+

